## __Aprendizaje no supervisado__

__Profesor__: Anthony D. Cho

__Ayudante__: Luis Oliveros

__Asunto__: Fuzzy C-Means

***

In [ ]:
## Instalación de libreria
#!python -m pip install ucimlrepo
#!python -m pip install fuzzy-c-means

In [ ]:
## Librerias
from ucimlrepo import fetch_ucirepo 
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from pandas import DataFrame

from sklearn.preprocessing import StandardScaler
from fcmeans import FCM
from sklearn.decomposition import PCA

## Carga de datos

__Dataset:__

Estos datos son referenciados en [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/109/wine) que son los resultados de un análisis químico de vinos cultivados en una misma región de Italia pero derivados de cultivos diferentes. 

<center>
    <img src='https://media.licdn.com/dms/image/v2/D5612AQHwzFJW75X22A/article-cover_image-shrink_720_1280/article-cover_image-shrink_720_1280/0/1674384291032?e=2147483647&v=beta&t=IBguDw6af4l1PyaTlV6Rw1YilLGFbUdn2_y178PYdww' width=800>
</center>

In [ ]:
## Cargar objeto de datos
dataset = fetch_ucirepo(id=109)

## Extraer los features del dataset
data = dataset.data.features 

## Mostrar los primeros 5 registros
display(data.head())

## Escalado de los datos
## Instancia del modelo de escalado
scaler = StandardScaler()

## Ajuste del modelo y transformación de los datos
data_scaled = scaler.fit_transform(data)
print('(shape) data: {}'.format(data_scaled.shape))

## Clase FCM

```{python}
    FCM(n_clsters=5, max_iter=150, m=2.0, error=1e-5)
```

| Parámetros | Descripción |
|------------|-------------|
| n_clusters | El número de clusteres a generar. |
| max_iter | Máxima cantidad de iteracioes. 
| m | Grado de difusor $m \in (1, \infty)$. |
| error | Tolerancia relativa entre iteraciones consecutivas de los centroides |
| random_state | (Optional): semilla de aleatoriedad |
| trained | Variable para almacenar si el modelo ha sido entrenado o no.|

<br>

| Atributos | Descripción |
|----------|-------------|
| center | Retorna los centros de los clusteres.|
| u | Retorna la matriz de pertenencia.|
| partition_coefficient | Retorna el valor de coeficiente de partitión y varía entre 0 y 1. Valor cercano a 1 mejor es la agrupación. |

<br>

|Funciones | Descripción |
|----------|-------------|
| fit(X) | Entrena el modelo con los parametros asignados.|
| predict(X) | Predice el cluster mas cercano a la que pertenece cada muestra. |
| soft_predict(X)  | Predice el nivel de pertenencia de cada observación a clusteres. |

In [ ]:
## Instancia del modelo
model = FCM(n_clusters=4, max_iter=1000, m=2.0, error=0.0005, random_state=9001)

## Ajuste del modelo
model.fit(data_scaled)

In [ ]:
## Mostrar los centros
print('Centros de los clusteres:')
print(model.centers)

In [ ]:
## Mostrar la matriz de pertenencia
print(model.u) 

In [ ]:
## Consulta en la matriz de pertenencia
print("Probabilidad de pertenencia para el primer punto")
print(model.u[0]) 

In [ ]:
## Identificar los grupos (etiqueta) a partir de la matriz de pertenencia u.
clusters = model.u.argmax(axis=1)
print("\nEtiqueta de cada punto")
print(clusters)

print("\nValor final del fuzzy partition coefficient: {}".format(model.partition_coefficient))

#### Buscando el valor de K

__Enfoque usando la medida fpc__

In [ ]:
## Maxima cantidad de clusteres
max_clusters = 20

## Almacenado de los valores de fuzzy partition coefficient
fpc_values = []

for n in tqdm(list(range(1, max_clusters+1))):

    ## Instancia del modelo
    model = FCM(n_clusters=n, max_iter=1000, m=2.0, error=0.0005, random_state=9001)

    ## Ajuste del modelo
    model.fit(data_scaled)

    ## Guardar el fpc obtenido
    fpc_values.append(model.partition_coefficient)


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(range(1, max_clusters+1), fpc_values)
plt.title('Elbow curve')
plt.xlabel('Numero de cluster')
plt.ylabel('FPC')
plt.tight_layout()
plt.show()

__Enfoque usando la medida de SSE__

In [ ]:
## Maxima cantidad de clusteres
max_clusters = 20

## Almacenado de los valores de SSE
SSE = []

for n in tqdm(list(range(1, max_clusters+1))):

    ## Instancia del modelo
    model = FCM(n_clusters=n, max_iter=1000, m=2.0, error=0.0005)

    ## Ajuste del modelo
    model.fit(data_scaled)

    ## Buscar las etiquetas de los clusteres para cada muestra
    clusters = model.predict(data_scaled)

    sse_value = 0
    for i in range(n):
        mask = (clusters == i)
        sse_value += ((data_scaled[mask, :] - model.centers[i])**2).sum()

    ## Guardar el SSE obtenido
    SSE.append(sse_value)


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(range(1, max_clusters+1), SSE)
plt.title('Elbow curve')
plt.xlabel('Numero de cluster')
plt.ylabel('SSE')
plt.tight_layout()
plt.show()

## Mejor modelo

In [ ]:
## Instancia del modelo
model = FCM(n_clusters=3, max_iter=1000, m=2.0, error=0.0005, random_state=9001)

## Ajuste del modelo
model.fit(data_scaled)

#### Visualización en 2D mediante PCA

In [ ]:
## Transformación de los datos usando PCA
pca_model = PCA(n_components=2)
pca_model.fit(data_scaled)
pca_data = pca_model.transform(data_scaled)

## Crear un dataframe con los datos transformados mediante PCA y agregar las etiquetas de los clusteres
pca_data = DataFrame(pca_data, columns=['PC1', 'PC2'])
pca_data['cluster'] = model.predict(data_scaled)

## Graficar los datos por clases. 
N = pca_data['cluster'].nunique()

plt.figure(figsize=(8, 6))
sns.scatterplot(data=pca_data, 
                x='PC1', y='PC2', 
                hue='cluster',
                palette=sns.color_palette()[:N])
plt.legend(loc=[1.01, 0.5], title='Cluster')
plt.tight_layout()
plt.show()